In [1]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [2]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

c:\Users\dhara\OneDrive\Desktop\House_Prediction\.venv\Scripts\python.exe
3.0.4
c:\Users\dhara\OneDrive\Desktop\House_Prediction\.venv\Lib\site-packages\xgboost\__init__.py


In [3]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

c:\Users\dhara\OneDrive\Desktop\House_Prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv("C:\\Users\\dhara\\OneDrive\\Desktop\\House_Prediction\\data\\processed\\feature_engineered_train.csv")
eval_df  = pd.read_csv("C:\\Users\\dhara\\OneDrive\\Desktop\\House_Prediction\\data\\processed\\feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [5]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [9]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================

from pathlib import Path
import mlflow
import optuna

# Set MLflow tracking directory in project root
mlruns_path = Path("mlruns").resolve()
mlruns_path.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(mlruns_path.as_uri())

# Set experiment
mlflow.set_experiment("xgboost_optuna_housing")

# Run Optuna study
study = optuna.create_study(direction="minimize")

study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)
print("Best value:", study.best_value)
print("MLflow Tracking URI:", mlflow.get_tracking_uri())

2026/11/02 17:23:53 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.
[I 2026-11-02 17:23:53,369] A new study created in memory with name: no-name-cf53f8b2-42cd-4d8d-9b62-511b33d22d9a
[I 2026-11-02 17:24:08,446] Trial 0 finished with value: 77850.45060141363 and parameters: {'n_estimators': 537, 'max_depth': 6, 'learning_rate': 0.17956891465130662, 'subsample': 0.8899971994672694, 'colsample_bytree': 0.8875374239169731, 'min_child_weight': 8, 'gamma': 2.038721762456894, 'reg_alpha': 0.0020715194221339935, 'reg_lambda': 3.168301741012172}. Best is trial 0 with value: 77850.45060141363.
[I 2026-11-02 17:24:15,741] Trial 1 finished with value: 76970.50504473888 and parameters: {'n_estimators': 299, 'max_depth': 5, 'learning_rate': 0.15148354863192023, 'subsample': 0.9304562994846876, 'colsample_bytree': 0.9115739219305671, 'min_child_weight': 10, 'gamma': 3.2024507963430753, 'reg_alpha': 3.949903822354514e-08, 'reg_lambda

Best params: {'n_estimators': 798, 'max_depth': 7, 'learning_rate': 0.012883757885937474, 'subsample': 0.6818941522109037, 'colsample_bytree': 0.5003700065021895, 'min_child_weight': 5, 'gamma': 2.3821479910270726, 'reg_alpha': 4.943396775937082e-06, 'reg_lambda': 2.376848310061922}
Best value: 69278.65878366288
MLflow Tracking URI: file:///C:/Users/dhara/OneDrive/Desktop/House_Prediction/notebooks/mlruns


In [10]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    mlflow.xgboost.log_model(best_model, name="model")

Final tuned model performance:
MAE: 32811.161826707284
RMSE: 69086.88925343132
R²: 0.963114870901472


c:\Users\dhara\OneDrive\Desktop\House_Prediction\.venv\Lib\site-packages\xgboost\sklearn.py:1028: UserWarning: [17:34:44] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1427: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  self.get_booster().save_model(fname)
2026/11/02 17:34:54 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/11/02 17:34:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
